# 00 — Smoke Tests

Validate that the full stack works from a notebook context:
DB connection, corpus loading, model artifacts, genre lookup,
similarity, clustering, engine, and metrics JSONL.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import joblib
import numpy as np
import polars as pl

# -- path setup
REPO_ROOT = Path("../..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

from paths import CACHE_DIR, DB_PATH, MODELS_DIR
from utils.logging import configure_logging

configure_logging()
print(f"REPO_ROOT: {REPO_ROOT}")
print(f"DB_PATH:   {DB_PATH} (exists={DB_PATH.exists()})")
print(f"MODELS_DIR:{MODELS_DIR} (exists={MODELS_DIR.exists()})")

## 1. DB Connection

In [ ]:
from etl.db import get_connection, init_schema

conn = get_connection(read_only=False)
init_schema(conn)

# Quick schema check
tables = conn.execute("SHOW TABLES").fetchall()
print(f"Tables: {[t[0] for t in tables]}")

# Sample from track_profile
sample = conn.execute("SELECT * FROM track_profile LIMIT 5").pl()
print(f"\ntrack_profile: {sample.shape}")
conn.close()
sample.head()

## 2. Corpus Load via `build_feature_matrix`

In [ ]:
from recommend.preprocessing import build_feature_matrix

conn_rw = get_connection(read_only=False)
init_schema(conn_rw)
corpus = build_feature_matrix(conn_rw)
conn_rw.close()

print(f"Corpus shape: {corpus.shape}")
print(f"Columns ({len(corpus.columns)}): {corpus.columns[:20]}...")
print(f"Dtypes sample: {dict(list(zip(corpus.columns[:10], corpus.dtypes[:10], strict=False)))}")

## 3. Cache Round-Trip

In [ ]:
cache_path = CACHE_DIR / "corpus_features.parquet"
if cache_path.exists():
    cached = pl.read_parquet(cache_path)
    print(f"Cached corpus: {cached.shape}")
    print(f"Match live corpus shape: {cached.shape == corpus.shape}")
else:
    print(f"Cache not found at {cache_path} — run train.py first")

## 4. Model Artifacts

In [ ]:
gmm_path = MODELS_DIR / "gmm_corpus.pkl"
scaler_path = MODELS_DIR / "scaler_corpus.pkl"

if gmm_path.exists() and scaler_path.exists():
    gmm = joblib.load(gmm_path)
    scaler = joblib.load(scaler_path)
    print(f"GMM: {gmm.n_components} components, covariance={gmm.covariance_type}")
    print(f"Scaler: {scaler.n_features_in_} features")

    # Quick predict on dummy row
    dummy = np.zeros((1, scaler.n_features_in_))
    probs = gmm.predict_proba(scaler.transform(dummy))
    print(f"Dummy predict_proba shape: {probs.shape}, sum={probs.sum():.4f}")
else:
    print("GMM/scaler not found — run train.py first")

## 5. Genre Lookup (ENOA)

In [ ]:
from recommend.modules.genre import genre_to_enoa, load_genre_map_from_db

conn_ro = get_connection(read_only=True)
genre_map = load_genre_map_from_db(conn_ro)
conn_ro.close()

print(f"Genre map: {len(genre_map)} genres")

test_genres = ["indie rock", "bossa nova", "deep house", "nonexistent genre"]
for g in test_genres:
    coords = genre_to_enoa(g, genre_map)
    print(f"  {g:25s} -> {coords}")

## 6. Similarity — `find_similar`

In [ ]:
from recommend.modules.similarity import SIMILARITY_FEATURES, find_similar

# Use first corpus row as query vector
query_row = corpus.row(0, named=True)
query_name = query_row.get("track_name", "unknown")
query_vector = np.array([float(query_row.get(f, 0)) for f in SIMILARITY_FEATURES])

results = find_similar(corpus, query_vector, k=5)
print(f"Query: {query_name}")
print("Top 5 similar:")
display_cols = ["track_name", "similarity_score"]
display_cols = [c for c in display_cols if c in results.columns]
results.select(display_cols).head()

## 7. Cluster Assignment

In [ ]:
from recommend.modules.clustering import build_cluster_features, predict_cluster_probs

if gmm_path.exists():
    sample_df = corpus.head(10)
    features, _ = build_cluster_features(sample_df, scaler, fit_scaler=False)
    probs = predict_cluster_probs(features, gmm)
    print(f"Cluster probs shape: {probs.shape}")
    print(f"Max prob per row: {probs.max(axis=1)}")
    print(f"Assigned clusters: {probs.argmax(axis=1)}")
else:
    print("Skipped — no GMM artifact")

## 8. Engine — Full Pipeline

In [ ]:
from recommend.engine import RecommendationEngine
from recommend.schemas import RecommendRequest

try:
    engine = RecommendationEngine()
    # Track-based recommendation using first corpus track
    first_id = corpus["id"][0]
    req = RecommendRequest(request_type="track", seed_id=first_id, k=5)
    result = engine.recommend(req)
    print(f"Pipeline: {result.pipeline_used}")
    print(f"Tracks: {result.track_names[:5]}")
    print(f"Scores: {[round(s, 4) for s in result.scores[:5]]}")
except Exception as exc:
    print(f"Engine init/recommend failed (expected if no artifacts): {exc}")

## 9. Metrics JSONL

In [ ]:
metrics_dir = MODELS_DIR / "metrics"
if metrics_dir.exists():
    jsonl_files = sorted(metrics_dir.glob("*.jsonl"))
    print(f"Found {len(jsonl_files)} metrics files:")
    for f in jsonl_files:
        lines = f.read_text().strip().split("\n")
        print(f"  {f.name}: {len(lines)} lines")
        # Show first entry
        if lines:
            first = json.loads(lines[0])
            print(f"    Sample: {first}")
else:
    print("No metrics dir — run train.py first")

---
✅ All smoke tests complete. If any cell above failed, fix the issue before proceeding to the other notebooks.